<a href="https://colab.research.google.com/github/componavt/topkar-space/blob/main/src/ner/bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import pandas as pd

df = pd.read_csv('abbreviations.csv')

# Группировка по расшифровкам с подсчетом уникальных аббревиатур
result = df.groupby('translation').agg({
    'abbreviation': 'count',
    'description': 'count'
}).rename(columns={'abbreviation': 'abbr_count', 'description': 'examples_count'})

# Сортировка по количеству примеров
result = result.sort_values('examples_count', ascending=False)

print(result)
result.to_csv('translation_stats.csv')

             abbr_count  examples_count
translation                            
река                164             164
село                158             158
остров              157             157
деревня             156             156
метров              146             146
посёлок              95              95
город                79              79
гора                 42              42
год                  35              35
мыс                  10              10
озеро                 5               5
севере                2               2


In [60]:
import pandas as pd
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset
from sklearn.utils import class_weight
import pickle
import os

# Загрузка и подготовка данных
df = pd.read_csv('abbreviations.csv')

# Удаляем классы с менее чем 2 примерами
class_counts = df['translation'].value_counts()
valid_classes = class_counts[class_counts >= 2].index
df = df[df['translation'].isin(valid_classes)]
print(f"Удалено классов с 1 примером: {len(class_counts) - len(valid_classes)}")
print(f"Осталось классов: {len(valid_classes)}")

# БАЛАНСИРОВКА для "г."
print("\n--- Балансировка классов для 'г.' ---")
g_examples = df[df['abbreviation'] == 'г.']
if len(g_examples) > 0:
    mountain_examples = g_examples[g_examples['translation'] == 'гора']
    city_examples = g_examples[g_examples['translation'] == 'город']

    print(f"Примеров 'г. = гора': {len(mountain_examples)}")
    print(f"Примеров 'г. = город': {len(city_examples)}")

    if len(mountain_examples) > 0 and len(city_examples) > 0:
        if len(mountain_examples) < len(city_examples):
            # Аугментация примеров с "горой"
            multiplier = len(city_examples) //len(mountain_examples) + 1
            mountain_examples_augmented = pd.concat([mountain_examples] * multiplier)
            df = pd.concat([df[df['abbreviation'] != 'г.'], city_examples, mountain_examples_augmented])
            print(f"Балансировка выполнена: примеров 'гора' увеличено до {len(mountain_examples_augmented)}")
        elif len(city_examples) < len(mountain_examples):
            # Аугментация примеров с "городом"
            multiplier = len(mountain_examples) // len(city_examples) + 1
            city_examples_augmented = pd.concat([city_examples] * multiplier)
            df = pd.concat([df[df['abbreviation'] != 'г.'], mountain_examples, city_examples_augmented])
            print(f"Балансировка выполнена: примеров 'город' увеличено до {len(city_examples_augmented)}")
        else:
            print("Классы уже сбалансированы")
else:
    print("Нет примеров с аббревиатурой 'г.'")

# Кодирование меток
le = LabelEncoder()
df['label'] = le.fit_transform(df['translation'])

# Токенизация
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

class AbbrDataset(Dataset):
    def __init__(self, texts, abbrs, labels):
        contexts = []
        for text, abbr in zip(texts, abbrs):
            if pd.isna(text):
                text = ""
            if abbr == 'г.':
                if 'под' in text.lower() or 'гора' in text.lower() or 'склон' in text.lower() or 'вершин' in text.lower():
                    context = f"ГОРА/ВОЗВЫШЕННОСТЬ: {text} [SEP] {abbr}"
                elif 'в' in text.lower() or 'жив' in text.lower() or 'насел' in text.lower():
                    context = f"НАСЕЛЕННЫЙ_ПУНКТ: {text} [SEP] {abbr}"
                else:
                    context = f"{text} [SEP] {abbr}"
            else:
                context = f"{text} [SEP] {abbr}"
            contexts.append(context)

        self.encodings = tokenizer(contexts, truncation=True, padding=True, max_length=128)
        self.labels = labels

    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()} | {'labels': torch.tensor(self.labels[idx])}

    def __len__(self): return len(self.labels)

# Разделение данных
X = df[['description', 'abbreviation']].values
y = df['label'].values

# Заполняем пропуски
X = np.array([[str(x[0]) if pd.notna(x[0]) else "", str(x[1]) if pd.notna(x[1]) else ""] for x in X])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Вычисление весов классов
unique_classes = np.unique(y_train)
if len(unique_classes) > 1:
    class_weights = class_weight.compute_class_weight('balanced', classes=unique_classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    print(f"\nВеса классов вычислены")
else:
    class_weights = torch.ones(len(unique_classes), dtype=torch.float)

# Создание датасетов
train_dataset = AbbrDataset(X_train[:,0], X_train[:,1], y_train)
test_dataset = AbbrDataset(X_test[:,0], X_test[:,1], y_test)

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nИспользуется устройство: {device}")

# Модель
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(le.classes_)
)
model.to(device)

# Кастомный Trainer с балансировкой
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if len(unique_classes) > 1:
            loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(model.device))
        else:
            loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Параметры обучения
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    eval_steps=100,
    save_steps=200,
    eval_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    save_total_limit=3,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

print("\n--- Начало обучения ---")
trainer.train()

# Функция предсказания
def predict(text, abbr):
    if pd.isna(text):
        text = ""

    # Подготовка контекста с балансировкой
    if abbr == 'г.':
        if any(word in text.lower() for word in ['под', 'гора', 'склон', 'вершин', 'восхожд', 'эльбрус', 'эверест']):
            context = f"ГОРА: {text} [SEP] {abbr}"
        elif any(word in text.lower() for word in ['в', 'жив', 'насел', 'москв', 'петерб']):
            context = f"ГОРОД: {text} [SEP] {abbr}"
        else:
            context = f"{text} [SEP] {abbr}"
    else:
        context = f"{text} [SEP] {abbr}"

    # Токенизация
    inputs = tokenizer(context, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Предсказание
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Получаем предсказание
    pred = outputs.logits.argmax().item()
    return le.inverse_transform([pred])[0]

# Сохранение
os.makedirs('./abbr_model_improved', exist_ok=True)
model.save_pretrained('./abbr_model_improved')
tokenizer.save_pretrained('./abbr_model_improved')
with open('./abbr_model_improved/label_encoder.pkl', 'wb') as f:
    pickle.dump(le.classes_, f)

print("\n--- Тестирование модели ---")

# Тест 1: Гора
test_text = "Бывшее болото под г. Горелиха."
test_abbr = "г."
print(f"\nТест 1: {test_text}")
print(f"Аббревиатура: {test_abbr}")
print(f"Предсказание: {predict(test_text, test_abbr)} (ожидается: гора)")

# Тест 2: Город
print(f"\nТест 2: г. Москва")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('г. Москва', 'г.')} (ожидается: город)")

# Тест 3: Гора
print(f"\nТест 3: под г. Эльбрус")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('под г. Эльбрус', 'г.')} (ожидается: гора)")

# Тест 4: Город
print(f"\nТест 4: живу в г. Санкт-Петербург")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('живу в г. Санкт-Петербург', 'г.')} (ожидается: город)")

# Тест 5: Гора
print(f"\nТест 5: восхождение на г. Эверест")
print(f"Аббревиатура: г.")
print(f"Предсказание: {predict('восхождение на г. Эверест', 'г.')} (ожидается: гора)")

# Статистика
print("\n--- Статистика после балансировки ---")
print(f"Всего классов: {len(le.classes_)}")
print(f"Всего примеров: {len(df)}")
# print(f"Примеров с 'г. = гора': {len(df[(df['abbreviation'] == 'г.') & (df['translation'] == 'гора)'])}")
# print(f"Примеров с 'г. = город': {len(df[(df['abbreviation'] == 'г.') & (df['translation'] == 'город')])}")

Удалено классов с 1 примером: 0
Осталось классов: 12

--- Балансировка классов для 'г.' ---
Примеров 'г. = гора': 42
Примеров 'г. = город': 79
Балансировка выполнена: примеров 'гора' увеличено до 84

Веса классов вычислены

Используется устройство: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



--- Начало обучения ---


Step,Training Loss,Validation Loss
100,1.804238,1.427266
200,0.629930,0.937768
300,0.501328,0.940849
400,0.475385,0.883799
500,0.247873,0.967032
600,0.157581,0.886758
700,0.359292,0.735342
800,0.231974,0.756141
900,0.253333,0.739001
1000,0.121636,0.748751


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Тестирование модели ---

Тест 1: Бывшее болото под г. Горелиха.
Аббревиатура: г.
Предсказание: гора (ожидается: гора)

Тест 2: г. Москва
Аббревиатура: г.
Предсказание: гора (ожидается: город)

Тест 3: под г. Эльбрус
Аббревиатура: г.
Предсказание: гора (ожидается: гора)

Тест 4: живу в г. Санкт-Петербург
Аббревиатура: г.
Предсказание: гора (ожидается: город)

Тест 5: восхождение на г. Эверест
Аббревиатура: г.
Предсказание: гора (ожидается: гора)

--- Статистика после балансировки ---
Всего классов: 11
Всего примеров: 1054


In [50]:
import torch
import pickle
from transformers import BertTokenizer, BertForSequenceClassification

# Загрузка одним блоком
def load_model(model_path='./abbr_model_improved'):
    tokenizer = BertTokenizer.from_pretrained(model_path)
    model = BertForSequenceClassification.from_pretrained(model_path)
    with open(f'{model_path}/label_encoder.pkl', 'rb') as f:
        classes = pickle.load(f)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.eval()

    return tokenizer, model, classes, device

# Инициализация при импорте
tokenizer, model, classes, device = load_model()

# Главная функция
def abbr(text, abbr):
    inputs = tokenizer(f"{text} [SEP] {abbr}", return_tensors="pt", max_length=128, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        pred = model(**inputs).logits.argmax().item()

    return classes[pred]

print(abbr("под г. Эльбрус", "г."))  # гора
print(abbr("Река, впадает в Белое море в г. Кемь", "г."))  # город
print(abbr("Мыс р. Кемь", "р."))  # река

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

гора
гора
река
